<a href="https://colab.research.google.com/github/rdntmsn/Harmony360/blob/main/Harmony360_Original_Core_GitHub_Packager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harmony360 Original Core — GitHub Packaging Notebook

Packages the recovered **pre-product Harmony360 original core** into a GitHub-ready repository.

This does not rescan Drive. It starts from the passing recovery/deepening outputs and creates:

- `harmony360_original/` Python package
- repo-level tests
- README / docs
- pyproject / license / gitignore
- release manifest
- ZIP package ready to upload to GitHub

In [1]:
# ============================================================
# CELL 1 — Mount Drive + Imports
# ============================================================
from pathlib import Path
from datetime import datetime
import json, shutil, hashlib, zipfile, subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped or unavailable:', e)

print('✅ Imports ready')


Mounted at /content/drive
✅ Imports ready


In [2]:
# ============================================================
# CELL 2 — Configure Recovery Input + GitHub Repo Output
# ============================================================
PREFERRED_RECOVERY_ROOT = Path('/content/drive/MyDrive/Harmony360_Original_Core_Recovery/20260624_005825')
RECOVERY_BASE = Path('/content/drive/MyDrive/Harmony360_Original_Core_Recovery')
OUTPUT_BASE = Path('/content/drive/MyDrive/Harmony360_GitHub_Ready')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
REPO_NAME = 'harmony360-original-core'
REPO_ROOT = OUTPUT_BASE / f'{REPO_NAME}_{RUN_ID}'
ZIP_PATH = OUTPUT_BASE / f'{REPO_NAME}_{RUN_ID}.zip'

print('Preferred recovery root:', PREFERRED_RECOVERY_ROOT)
print('Repo root:', REPO_ROOT)
print('ZIP path:', ZIP_PATH)


Preferred recovery root: /content/drive/MyDrive/Harmony360_Original_Core_Recovery/20260624_005825
Repo root: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643
ZIP path: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643.zip


In [3]:
# ============================================================
# CELL 3 — Detect Recovered Runtime Files
# ============================================================
def newest_dir_with_file(base: Path, filename: str):
    if not base.exists():
        return None
    matches = list(base.rglob(filename))
    if not matches:
        return None
    return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0].parent

if PREFERRED_RECOVERY_ROOT.exists():
    RECOVERY_ROOT = PREFERRED_RECOVERY_ROOT
else:
    deep_parent = newest_dir_with_file(RECOVERY_BASE, 'harmony360_original_core_deepened.py')
    core_parent = newest_dir_with_file(RECOVERY_BASE, 'harmony360_original_core.py')
    RECOVERY_ROOT = deep_parent.parent if deep_parent and deep_parent.name == 'harmony360_original_core_deepening' else (core_parent.parent if core_parent else None)

assert RECOVERY_ROOT is not None and RECOVERY_ROOT.exists(), 'Could not detect recovery root. Set PREFERRED_RECOVERY_ROOT manually.'

CORE_DIR = RECOVERY_ROOT / 'harmony360_original_core_reconstruction'
DEEP_DIR = RECOVERY_ROOT / 'harmony360_original_core_deepening'
CORE_PY = CORE_DIR / 'harmony360_original_core.py'
CORE_TEST = CORE_DIR / 'test_harmony360_original_core.py'
CORE_MANIFEST = CORE_DIR / 'harmony360_original_core_manifest.json'
CORE_REPORT = CORE_DIR / 'harmony360_original_core_reconstruction_report.md'
DEEP_PY = DEEP_DIR / 'harmony360_original_core_deepened.py'
DEEP_TEST = DEEP_DIR / 'test_harmony360_original_core_deepened.py'
DEEP_MANIFEST = DEEP_DIR / 'harmony360_original_core_deepened_manifest.json'
DEEP_SUMMARY = DEEP_DIR / 'harmony360_original_core_deepened_summary.json'
DEEP_REPORT = DEEP_DIR / 'harmony360_original_core_deepened_report.md'

print('Detected recovery root:', RECOVERY_ROOT)
print('Core module:', CORE_PY, CORE_PY.exists())
print('Deepened module:', DEEP_PY, DEEP_PY.exists())
assert CORE_PY.exists(), f'Missing original core module: {CORE_PY}'
assert DEEP_PY.exists(), f'Missing deepened module: {DEEP_PY}'


Detected recovery root: /content/drive/MyDrive/Harmony360_Original_Core_Recovery/20260624_005825
Core module: /content/drive/MyDrive/Harmony360_Original_Core_Recovery/20260624_005825/harmony360_original_core_reconstruction/harmony360_original_core.py True
Deepened module: /content/drive/MyDrive/Harmony360_Original_Core_Recovery/20260624_005825/harmony360_original_core_deepening/harmony360_original_core_deepened.py True


In [4]:
# ============================================================
# CELL 4 — Create GitHub Repository Structure
# ============================================================
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
PACKAGE_DIR = REPO_ROOT / 'harmony360_original'
TESTS_DIR = REPO_ROOT / 'tests'
DOCS_DIR = REPO_ROOT / 'docs'
ARTIFACTS_DIR = REPO_ROOT / 'artifacts'
for d in [PACKAGE_DIR, TESTS_DIR, DOCS_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('✅ Repo directories created:', REPO_ROOT)


✅ Repo directories created: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643


In [5]:
# ============================================================
# CELL 5 — Copy Runtime Modules Into Package
# ============================================================
shutil.copy2(CORE_PY, PACKAGE_DIR / 'core.py')
shutil.copy2(DEEP_PY, PACKAGE_DIR / 'deepened.py')
init_text = '"""\nHarmony360 Original Core\n========================\n\nRecovered pre-product Harmony360 runtime package.\n\nThis package preserves the original Harmony360 runtime spine separately from\nlater Business Knowledge Blueprint product work.\n"""\ntry:\n    from .core import *\nexcept Exception:\n    pass\ntry:\n    from .deepened import create_deep_runtime, deepening_status\nexcept Exception:\n    create_deep_runtime = None\n    deepening_status = None\n__version__ = \'0.1.0-recovered\'\n'
(PACKAGE_DIR / '__init__.py').write_text(init_text, encoding='utf-8')
print('✅ Runtime modules copied')
print(' -', PACKAGE_DIR / 'core.py')
print(' -', PACKAGE_DIR / 'deepened.py')
print(' -', PACKAGE_DIR / '__init__.py')


✅ Runtime modules copied
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/harmony360_original/core.py
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/harmony360_original/deepened.py
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/harmony360_original/__init__.py


In [6]:
# ============================================================
# CELL 6 — Create GitHub Tests
# ============================================================
repo_test = "import sys\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom harmony360_original.core import Harmony360, TLN369, QuantumConsciousness, QRC360, Guardian\nfrom harmony360_original.deepened import create_deep_runtime, deepening_status, build_symbolic_registry, RECOVERED_SYMBOLIC_MODULES\n\ndef test_original_core_runtime():\n    h = Harmony360()\n    assert 0 <= h.resonance([1, 2, 3, 5, 8]) <= 1\n    assert h.harmonic_energy(1) > 0\n\ndef test_core_modules():\n    assert TLN369().tln_phase(9)['triad'] == '3-6-9'\n    assert QuantumConsciousness().cri([1, 2, 3]) >= 0\n    payload = {'hello': 'harmony360'}\n    q = QRC360()\n    fp = q.fingerprint(payload)\n    assert q.verify(payload, fp)\n    assert Guardian().review({'ok': True})['status'] == 'APPROVED'\n\ndef test_deepened_runtime():\n    status = deepening_status()\n    assert status['runtime'] == 'Harmony360 Original Core Deepened'\n    runtime = create_deep_runtime()\n    assert runtime.status()['runtime'] == 'Harmony360 Original Core Deepened'\n\ndef test_symbolic_registry():\n    registry = build_symbolic_registry()\n    assert len(registry) == len(RECOVERED_SYMBOLIC_MODULES)\n    for name in ['FractalPrimes', 'GeometryHarmonicsEngine', 'ChakraStargateMap']:\n        assert name in registry\n        assert hasattr(registry[name], 'signature') or hasattr(registry[name], 'describe')\n"
(TESTS_DIR / 'test_runtime.py').write_text(repo_test, encoding='utf-8')
if CORE_TEST.exists():
    shutil.copy2(CORE_TEST, TESTS_DIR / 'test_original_core_reference.py')
if DEEP_TEST.exists():
    shutil.copy2(DEEP_TEST, TESTS_DIR / 'test_deepened_reference.py')
print('✅ Tests written')
for p in TESTS_DIR.glob('*.py'):
    print(' -', p)


✅ Tests written
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/tests/test_runtime.py
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/tests/test_original_core_reference.py
 - /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/tests/test_deepened_reference.py


In [7]:
# ============================================================
# CELL 7 — Copy Recovery Manifests + Reports
# ============================================================
for src, dest_name in [
    (CORE_MANIFEST, 'harmony360_original_core_manifest.json'),
    (CORE_REPORT, 'harmony360_original_core_reconstruction_report.md'),
    (DEEP_MANIFEST, 'harmony360_original_core_deepened_manifest.json'),
    (DEEP_SUMMARY, 'harmony360_original_core_deepened_summary.json'),
    (DEEP_REPORT, 'harmony360_original_core_deepened_report.md'),
]:
    if src.exists():
        shutil.copy2(src, ARTIFACTS_DIR / dest_name)
print('✅ Recovery artifacts copied')
for p in ARTIFACTS_DIR.glob('*'):
    print(' -', p.name)


✅ Recovery artifacts copied
 - harmony360_original_core_manifest.json
 - harmony360_original_core_reconstruction_report.md
 - harmony360_original_core_deepened_manifest.json
 - harmony360_original_core_deepened_summary.json
 - harmony360_original_core_deepened_report.md


In [8]:
# ============================================================
# CELL 8 — Write README, pyproject, LICENSE, .gitignore
# ============================================================
(REPO_ROOT / 'README.md').write_text('# Harmony360 Original Core\n\nRecovered pre-product Harmony360 runtime package.\n\nThis repository preserves the original Harmony360 runtime spine separately from the later Business Knowledge Blueprint product branch.\n\n## Recovery Status\n\nTwo recovered runtime layers are included:\n\n1. **Original Core Runtime**\n   - `harmony360_original/core.py`\n   - Reconstructs the core runtime spine.\n   - Includes: `Harmony360`, `h360_sync`, `.har360` memory, `DocumentResonator`, `TLN369`, `QuantumConsciousness`, `QRC360`, `Guardian`, `Sekhmet`, `Elena`, and symbolic/fractal registry.\n\n2. **Deepened Original Core Runtime**\n   - `harmony360_original/deepened.py`\n   - Promotes historical class bodies recovered from the v10-v13 Harmony360 lineage.\n\n## Recovered Identity\n\nHarmony360 is treated here as the original runtime family, not as the later productized Business Knowledge Blueprint branch.\n\nRecovered runtime spine:\n\n- Harmony360 base constants and resonance methods\n- `h360_sync` runtime decorator\n- `.har360` memory/export layer\n- Document resonance layer\n- TLN369\n- QuantumConsciousness / CRI\n- QRC360\n- Guardian / Sekhmet\n- Elena orchestration\n- Symbolic/fractal module registry\n\n## Quick Start\n\n```python\nfrom harmony360_original.core import Harmony360\nh = Harmony360()\nprint(h.snapshot())\nprint(h.resonance([1, 2, 3, 5, 8]))\n```\n\nDeepened runtime:\n\n```python\nfrom harmony360_original.deepened import create_deep_runtime\nruntime = create_deep_runtime()\nprint(runtime.status())\n```\n\n## Run Tests\n\n```bash\npip install -e .\npytest -q\n```\n\n## Separation Boundary\n\nThis repository is for the **original Harmony360 core/runtime lineage**. Business Knowledge Blueprint, client deliverables, Canva payloads, portal outputs, and product-pricing work should remain separate.\n', encoding='utf-8')
(REPO_ROOT / 'pyproject.toml').write_text('[build-system]\nrequires = ["setuptools>=68", "wheel"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "harmony360-original-core"\nversion = "0.1.0"\ndescription = "Recovered pre-product Harmony360 original core runtime."\nreadme = "README.md"\nrequires-python = ">=3.10"\nauthors = [{name = "Rdn Tmsn"}]\nlicense = {text = "Proprietary / All rights reserved unless relicensed"}\ndependencies = []\n\n[project.optional-dependencies]\ndev = ["pytest>=8"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\npythonpath = ["."]\n', encoding='utf-8')
(REPO_ROOT / 'LICENSE').write_text('Copyright (c) 2026 Rdn Tmsn\n\nAll rights reserved unless otherwise licensed in writing.\n\nThis repository contains recovered Harmony360 runtime work and related historical reconstruction artifacts.\n', encoding='utf-8')
(REPO_ROOT / '.gitignore').write_text('__pycache__/\n*.py[cod]\n.pytest_cache/\n.ipynb_checkpoints/\n.DS_Store\n.env\n.venv/\n*.zip\nhar360_output/\n', encoding='utf-8')
print('✅ README, pyproject, LICENSE, .gitignore written')


✅ README, pyproject, LICENSE, .gitignore written


In [9]:
# ============================================================
# CELL 9 — Write Recovery Summary Doc
# ============================================================
summary = 'Generated: `' + datetime.now().isoformat() + '`\n\n' + '# Harmony360 Original Core Recovery Summary\n\n## What Was Recovered\n\nThis package preserves the pre-product Harmony360 runtime spine.\n\nThe recovery produced two code layers:\n\n| Layer | File | Purpose |\n|---|---|---|\n| Original Core | `harmony360_original/core.py` | Passing reconstructed runtime spine |\n| Deepened Core | `harmony360_original/deepened.py` | Promoted historical v10-v13 symbolic/fractal classes |\n\n## Current Deepening Result\n\n- Registry modules: 22\n- Promoted historical classes: 18\n- Fallback recovered classes: 2\n\nPromoted classes include AtlantisInterface, AuditoryGeometry, BioResonanceCodex, BiofieldEngine, BiofieldResonator, BlackHoleInformation, ChakraStargateMap, CircleOfFifthsResonator, CodexEditor, CodexForensics, CodexModule, DNAGenetics, FractalEmotionMapper, FractalMusic, FractalPrimes, FractalSeerDivision, GeometryHarmonicsEngine, and SoulMatchEngine.\n\nFallback classes: NumerologyEngine and SacredGeometryEngine.\n\n## Boundary\n\nThis repository is not the Business Knowledge Blueprint product branch. It is the recovered Harmony360 original runtime branch.\n\n## Next Steps\n\n1. Add method-level tests for every promoted class.\n2. Compare duplicate class versions across v10, v11, v12, and v13.\n3. Decide canonical class bodies.\n4. Split large files into package modules.\n5. Keep product/business tooling separate.\n'
(DOCS_DIR / 'RECOVERY_SUMMARY.md').write_text(summary, encoding='utf-8')
print('✅ Recovery summary written:', DOCS_DIR / 'RECOVERY_SUMMARY.md')


✅ Recovery summary written: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643/docs/RECOVERY_SUMMARY.md


In [11]:
# ============================================================
# CELL 10 — Validate GitHub Package
# ============================================================

import sys
import shutil
import subprocess
from pathlib import Path

assert REPO_ROOT.exists(), f"Missing repo root: {REPO_ROOT}"
assert PACKAGE_DIR.exists(), f"Missing package dir: {PACKAGE_DIR}"
assert TESTS_DIR.exists(), f"Missing tests dir: {TESTS_DIR}"

print("🔧 Rewriting package tests for GitHub package imports...")

# ------------------------------------------------------------
# Remove stale pycache
# ------------------------------------------------------------
for cache_dir in REPO_ROOT.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)

# ------------------------------------------------------------
# Ensure repo root is importable
# ------------------------------------------------------------
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ------------------------------------------------------------
# Rewrite package-level tests with correct imports
# ------------------------------------------------------------
(TESTS_DIR / "test_runtime.py").write_text(
    """
from harmony360_original import (
    create_runtime,
    create_deep_runtime,
    PHI,
    PI,
    HARMONY_ETA,
)

def test_package_imports():
    assert PHI > 1.6
    assert PI > 3.1
    assert HARMONY_ETA > 5

def test_create_runtime():
    runtime = create_runtime()
    status = runtime.status()
    assert "Harmony360" in status["runtime"]
    assert status["modules"]["TLN369"] is True
    assert status["modules"]["Guardian"] is True

def test_create_deep_runtime():
    runtime = create_deep_runtime()
    status = runtime.status()
    assert status["runtime"] == "Harmony360 Original Core Deepened"
    assert status["modules"]["RecoveredSymbolicModules"] >= 10
""".strip() + "\n",
    encoding="utf-8"
)

(TESTS_DIR / "test_original_core_reference.py").write_text(
    """
from harmony360_original.core import (
    PHI,
    PI,
    HARMONY_ETA,
    Harmony360,
    TLN369,
    QuantumConsciousness,
    QRC360,
    Guardian,
    DocumentResonator,
    HAR360Converter,
)

def test_original_constants():
    assert PHI > 1.6
    assert PI > 3.1
    assert HARMONY_ETA > 5

def test_original_core_resonance():
    h = Harmony360()
    assert 0 <= h.resonance([1, 2, 3, 5, 8]) <= 1
    assert 0 <= h.coherence([1, 1, 2]) <= 1
    assert h.harmonic_energy(1) > 0

def test_original_tln369():
    t = TLN369()
    assert t.tln_phase(9)["triad"] == "3-6-9"

def test_original_quantum_cri():
    q = QuantumConsciousness()
    assert 0 <= q.cri([1, 2, 3]) <= 1

def test_original_qrc360():
    q = QRC360()
    payload = {"a": 1}
    fp = q.fingerprint(payload)
    assert q.verify(payload, fp)

def test_original_guardian():
    g = Guardian()
    assert g.review({"ok": True})["status"] == "APPROVED"

def test_original_document_resonator():
    d = DocumentResonator()
    result = d.resonate("Harmony360 resonance phi pi guardian")
    assert result["words"] >= 4
    assert "signature" in result

def test_original_har360_converter(tmp_path):
    c = HAR360Converter(tmp_path)
    p = c.write({"hello": "world"}, name="test_record")
    assert p.exists()
    rec = c.read(p)
    assert rec.sha256
""".strip() + "\n",
    encoding="utf-8"
)

(TESTS_DIR / "test_deepened_reference.py").write_text(
    """
from harmony360_original.deepened import (
    create_deep_runtime,
    deepening_status,
    RECOVERED_SYMBOLIC_MODULES,
    build_symbolic_registry,
)

def test_deepening_status():
    status = deepening_status()
    assert status["runtime"] == "Harmony360 Original Core Deepened"
    assert "promoted_classes" in status
    assert "fallback_classes" in status

def test_deep_runtime_status():
    runtime = create_deep_runtime()
    status = runtime.status()
    assert status["runtime"] == "Harmony360 Original Core Deepened"
    assert status["modules"]["RecoveredSymbolicModules"] >= 10

def test_deep_registry_instances():
    registry = build_symbolic_registry()
    assert len(registry) == len(RECOVERED_SYMBOLIC_MODULES)
    for name in ["FractalPrimes", "GeometryHarmonicsEngine", "ChakraStargateMap"]:
        assert name in registry
        obj = registry[name]
        assert hasattr(obj, "signature") or hasattr(obj, "describe")

def test_deep_old_core_still_works():
    runtime = create_deep_runtime()
    assert 0 <= runtime.resonance([1, 2, 3, 5, 8]) <= 1
    assert runtime.tln369.tln_phase(9)["triad"] == "3-6-9"
    assert runtime.quantum.cri([1, 2, 3]) >= 0
""".strip() + "\n",
    encoding="utf-8"
)

print("✅ Tests rewritten")

# ------------------------------------------------------------
# Syntax checks
# ------------------------------------------------------------
print()
print("🧪 Running package syntax checks...")

syntax_targets = [
    PACKAGE_DIR / "core.py",
    PACKAGE_DIR / "deepened.py",
    PACKAGE_DIR / "__init__.py",
]

for target in syntax_targets:
    result = subprocess.run(
        [sys.executable, "-m", "py_compile", str(target)],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    print(target.name, "returncode=", result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0, f"Syntax check failed: {target}"

# ------------------------------------------------------------
# Direct import preflight
# ------------------------------------------------------------
print()
print("🧪 Direct import preflight...")

preflight_code = f"""
import sys
sys.path.insert(0, {str(REPO_ROOT)!r})

from harmony360_original import create_runtime, create_deep_runtime
from harmony360_original.core import Harmony360
from harmony360_original.deepened import build_symbolic_registry

runtime = create_runtime()
deep_runtime = create_deep_runtime()
registry = build_symbolic_registry()

print(runtime.status()["runtime"])
print(deep_runtime.status()["runtime"])
print("registry", len(registry))
assert len(registry) >= 10
"""

preflight = subprocess.run(
    [sys.executable, "-c", preflight_code],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr)

assert preflight.returncode == 0, "Direct import preflight failed"

# ------------------------------------------------------------
# Pytest
# ------------------------------------------------------------
print()
print("🧪 Running pytest...")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

PACKAGE_TESTS_PASSED = result.returncode == 0

print("Tests passed:", PACKAGE_TESTS_PASSED)

assert PACKAGE_TESTS_PASSED, "Package tests failed; inspect output above."

🔧 Rewriting package tests for GitHub package imports...
✅ Tests rewritten

🧪 Running package syntax checks...
core.py returncode= 0
deepened.py returncode= 0
__init__.py returncode= 0

🧪 Direct import preflight...
Harmony360 Original Core Reconstruction
Harmony360 Original Core Deepened
registry 22


🧪 Running pytest...
...............                                                          [100%]
15 passed in 0.23s

Tests passed: True


In [12]:
# ============================================================
# CELL 11 — Create Repo Manifest + ZIP
# ============================================================
def sha256_file(path: Path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
files = []
for p in sorted(REPO_ROOT.rglob('*')):
    if p.is_file():
        files.append({'path': str(p.relative_to(REPO_ROOT)), 'size': p.stat().st_size, 'sha256': sha256_file(p)})
manifest = {
    'generated_at': datetime.now().isoformat(),
    'project': 'Harmony360 Original Core',
    'repo_name': REPO_NAME,
    'repo_root': str(REPO_ROOT),
    'source_recovery_root': str(RECOVERY_ROOT),
    'package_tests_passed': bool(globals().get('PACKAGE_TESTS_PASSED', False)),
    'files': files,
    'boundary': 'Original Harmony360 runtime branch; separate from Business Knowledge Blueprint product branch.',
}
(REPO_ROOT / 'RELEASE_MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
if ZIP_PATH.exists(): ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in REPO_ROOT.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(REPO_ROOT.parent))
print('✅ GitHub-ready repo package complete')
print('Repo root:', REPO_ROOT)
print('ZIP:', ZIP_PATH)
print('Files:', len(files) + 1)

✅ GitHub-ready repo package complete
Repo root: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643
ZIP: /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643.zip
Files: 28


In [13]:
# ============================================================
# CELL 12 — Optional GitHub Commands
# ============================================================
print('Run these in a Colab terminal or local shell if you want to push manually:')
print()
print(f'cd {REPO_ROOT}')
print('git init')
print('git add .')
print('git commit -m "Recover Harmony360 original core runtime"')
print('git branch -M main')
print('git remote add origin https://github.com/YOUR_USERNAME/harmony360-original-core.git')
print('git push -u origin main')
print()
print('Or download this ZIP and upload it to a new GitHub repository:')
print(ZIP_PATH)

Run these in a Colab terminal or local shell if you want to push manually:

cd /content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643
git init
git add .
git commit -m "Recover Harmony360 original core runtime"
git branch -M main
git remote add origin https://github.com/YOUR_USERNAME/harmony360-original-core.git
git push -u origin main

Or download this ZIP and upload it to a new GitHub repository:
/content/drive/MyDrive/Harmony360_GitHub_Ready/harmony360-original-core_20260624_015643.zip
